In [3]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

# Load datasets
train = pd.read_csv("train.csv")
store = pd.read_csv("store.csv")

# Merge datasets
data = pd.merge(train, store, on="Store", how="left")

# Keep only open stores
data = data[data["Open"] == 1].copy()

# Convert Date to datetime
data["Date"] = pd.to_datetime(data["Date"])

# Fill missing values
data["CompetitionDistance"] = data["CompetitionDistance"].fillna(
    data["CompetitionDistance"].median()
)
data["CompetitionOpenSinceMonth"] = data["CompetitionOpenSinceMonth"].fillna(0)
data["CompetitionOpenSinceYear"] = data["CompetitionOpenSinceYear"].fillna(0)
data["Promo2"] = data["Promo2"].fillna(0)
data["Promo2SinceWeek"] = data["Promo2SinceWeek"].fillna(0)
data["Promo2SinceYear"] = data["Promo2SinceYear"].fillna(0)
data["PromoInterval"] = data["PromoInterval"].fillna("None")

# Feature engineering
data["Year"] = data["Date"].dt.year
data["Month"] = data["Date"].dt.month
data["Day"] = data["Date"].dt.day

# Encode categorical variables
data["StoreType"] = data["StoreType"].astype("category").cat.codes
data["Assortment"] = data["Assortment"].astype("category").cat.codes
data["StateHoliday"] = data["StateHoliday"].astype("category").cat.codes
data["PromoInterval"] = data["PromoInterval"].astype("category").cat.codes

# Sort by store and time before creating lag features
data = data.sort_values(by=["Store", "Year", "Month", "Day"])

# Create lag features
data["Lag_1"] = data.groupby("Store")["Sales"].shift(1)
data["Lag_7"] = data.groupby("Store")["Sales"].shift(7)

# Remove rows with missing lag values
data = data.dropna()

# Drop columns not used for training
data = data.drop(["Date", "Customers"], axis=1)

# Define target and features
y = data["Sales"]
X = data.drop("Sales", axis=1)

# Time-based split
X = X.sort_values(by=["Year", "Month", "Day"])
y = y.loc[X.index]

split_index = int(len(X) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

# Train final model
model = RandomForestRegressor(
    n_estimators=20,
    max_depth=10,
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Evaluation
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Final Model: Random Forest with Lag Features")
print("MAE:", mae)
print("R2 Score:", r2)

# Save model
joblib.dump(model, "rossmann_rf_lag.pkl")
print("rossmann_rf_lag.pkl")

C:\Users\HP\AppData\Local\Temp\ipykernel_21028\2992140889.py:7: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv("train.csv")


Final Model: Random Forest with Lag Features
MAE: 866.0364862003341
R2 Score: 0.8307700834638987
rossmann_rf_lag.pkl
